In [22]:
# Import pandas library for data manipulation and transformation
import pandas as pd


# Load raw NFL player data from the Bronze layer in the Fabric Lakehouse
# This file was created during the Sleeper API ingestion process
players = pd.read_json("/lakehouse/default/Files/players_nfl.json")


# Preview raw player data before transformation
players.head()


# The Sleeper API returns players as a dictionary where player IDs are keys.
# Transpose the dataset so each player becomes a row and player attributes become columns.
# This creates a structure more suitable for analytics and dimensional modeling.
players = players.T.reset_index()


# Preview transformed player dimension structure
players.head()


StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 46, Finished, Available, Finished, False)

/tmp/ipykernel_9918/550638496.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  players = pd.read_json("/lakehouse/default/Files/players_nfl.json")
/tmp/ipykernel_9918/550638496.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  players = pd.read_json("/lakehouse/default/Files/players_nfl.json")
/tmp/ipykernel_9918/550638496.py:7: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime s

,index,age,weight,birth_city,high_school,first_name,birth_country,height,team,last_name,...,sport,position,news_updated,pandascore_id,player_shard,yahoo_id,search_rank,gsis_id,team_abbr,practice_description
0,6462,26,245,None,Douglas County,Ellis,None,75,None,Richardson,...,nfl,TE,None,None,8,32262,9999999,00-0035057,None,None
1,11255,None,306,None,Davis (CA),Nick,None,74,None,Amoah,...,nfl,OL,None,None,5,None,9999999,None,None,None
2,8842,None,186,None,Iona Prep (NY),Malkelm,None,70,None,Morrison,...,nfl,CB,1653351613345,None,0,None,9999999,None,None,None
3,13940,None,305,None,Salpointe Catholic (AZ),Bruno,None,77,BUF,Fina,...,nfl,OL,None,None,5,None,9999999,None,None,None
4,7926,24,250,None,Hough (NC),Carl,None,74,None,Tucker,...,nfl,TE,1631568053348,None,7,None,426,None,None,None


In [23]:
# Select only relevant player attributes needed for the player dimension table
# Removing unnecessary fields from the raw Sleeper player dataset
dim_player = players[
    [
        "player_id",
        "first_name",
        "last_name",
        "position",
        "team",
        "active",
    ]
]


# Preview the transformed player dimension table
dim_player.head()

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 47, Finished, Available, Finished, False)

,player_id,first_name,last_name,position,team,active
0,6462,Ellis,Richardson,TE,None,True
1,11255,Nick,Amoah,OL,None,True
2,8842,Malkelm,Morrison,CB,None,True
3,13940,Bruno,Fina,OL,BUF,True
4,7926,Carl,Tucker,TE,None,True


In [24]:
# Convert Pandas dataframe into a Spark dataframe.
# Spark is used by Fabric to create and manage Delta tables.
spark_df = spark.createDataFrame(dim_player)


# Save the player dimension as a Delta table in the Fabric Lakehouse.
# Delta format provides reliable storage, schema management, and performance benefits.
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Dim_Player")

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 48, Finished, Available, Finished, False)

In [25]:
# Import pandas library for data loading and transformation
import pandas as pd


# Load raw owner/user data from the Bronze layer in the Fabric Lakehouse
# These files were created during the Sleeper API ingestion process
users_2024 = pd.read_json("/lakehouse/default/Files/users_2024.json")
users_2025 = pd.read_json("/lakehouse/default/Files/users_2025.json")


# Preview 2024 owner data before transformation
users_2024.head()

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 49, Finished, Available, Finished, False)

,avatar,display_name,is_bot,is_owner,league_id,metadata,settings,user_id
0,49f969224407758685e0a113d202d9e1,WhiteIversonn,False,True,1072575253525798912,"{'allow_pn': 'on', 'archived': 'off', 'avatar'...",NaN,600417631085854720
1,b8ad6de3e6370e3956108e78dd78f960,alxlddy,False,True,1072575253525798912,"{'allow_pn': 'on', 'mention_pn': 'on'}",NaN,600732404100956160
2,bcc66d3988fda416ddccb76145a9c187,Shotier1,False,False,1072575253525798912,"{'allow_pn': 'on', 'mention_pn': 'on'}",NaN,728639823287574528
3,47cf806786b78560f76d325af3bac68b,prezofzynbabwe,False,False,1072575253525798912,"{'allow_pn': 'on', 'archived': 'off', 'mention...",NaN,730546204496211968
4,4f4090e5e9c3941414db40a871e3e909,Lovegun69,False,False,1072575253525798912,"{'allow_pn': 'on', 'avatar': 'https://sleeperc...",NaN,734418867744526336


In [26]:
#combine the seasons!

users = pd.concat(
    [users_2024, users_2025],
    ignore_index=True
)

users.head()

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 50, Finished, Available, Finished, False)

,avatar,display_name,is_bot,is_owner,league_id,metadata,settings,user_id
0,49f969224407758685e0a113d202d9e1,WhiteIversonn,False,1.0,1072575253525798912,"{'allow_pn': 'on', 'archived': 'off', 'avatar'...",NaN,600417631085854720
1,b8ad6de3e6370e3956108e78dd78f960,alxlddy,False,1.0,1072575253525798912,"{'allow_pn': 'on', 'mention_pn': 'on'}",NaN,600732404100956160
2,bcc66d3988fda416ddccb76145a9c187,Shotier1,False,0.0,1072575253525798912,"{'allow_pn': 'on', 'mention_pn': 'on'}",NaN,728639823287574528
3,47cf806786b78560f76d325af3bac68b,prezofzynbabwe,False,0.0,1072575253525798912,"{'allow_pn': 'on', 'archived': 'off', 'mention...",NaN,730546204496211968
4,4f4090e5e9c3941414db40a871e3e909,Lovegun69,False,0.0,1072575253525798912,"{'allow_pn': 'on', 'avatar': 'https://sleeperc...",NaN,734418867744526336


In [27]:
users.columns

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 51, Finished, Available, Finished, False)

Index(['avatar', 'display_name', 'is_bot', 'is_owner', 'league_id', 'metadata',
       'settings', 'user_id'],
      dtype='object')

In [28]:
#choose columns we want
dim_owner = users[
    [
        "user_id",
        "display_name"
    ]
].copy()

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 52, Finished, Available, Finished, False)

In [29]:
#remove dupliates
dim_owner = dim_owner.drop_duplicates(subset=["user_id"])

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 53, Finished, Available, Finished, False)

In [30]:
dim_owner = dim_owner.astype("string")

dim_owner.head(12)

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 54, Finished, Available, Finished, False)

,user_id,display_name
0,600417631085854720,WhiteIversonn
1,600732404100956160,alxlddy
2,728639823287574528,Shotier1
3,730546204496211968,prezofzynbabwe
4,734418867744526336,Lovegun69
5,867819528762212352,BennyBlanco401
6,870100125954015232,duranduran12
7,870356288188678144,Evcali15
8,999120491128528896,Dnasty222
9,999125378960617472,StvLddy


In [31]:
# Convert Pandas dataframe into a Spark dataframe.
# Spark is used by Fabric to create and manage Delta tables.
spark_df = spark.createDataFrame(dim_owner)


# Save the owner dimension as a Delta table in the Fabric Lakehouse.
# Overwrite ensures the table is refreshed with the latest transformed data.
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Dim_Owner")


StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 55, Finished, Available, Finished, False)

In [32]:
import pandas as pd

# Read roster files
rosters_2024 = pd.read_json("/lakehouse/default/Files/rosters_2024.json")
rosters_2025 = pd.read_json("/lakehouse/default/Files/rosters_2025.json")

# Add season metadata
rosters_2024["season"] = 2024
rosters_2025["season"] = 2025

# Combine years
rosters = pd.concat(
    [rosters_2024, rosters_2025],
    ignore_index=True
)

# Create dimension table
dim_roster = rosters[
    [
        "season",
        "roster_id",
        "owner_id"
    ]
].copy()

# Clean data types
dim_roster = dim_roster.astype("string")

# Preview
dim_roster.head()

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 56, Finished, Available, Finished, False)

,season,roster_id,owner_id
0,2024,1,600417631085854720
1,2024,2,600732404100956160
2,2024,3,734418867744526336
3,2024,4,999125378960617472
4,2024,5,730546204496211968


In [33]:
# Create a unique roster identifier by combining season and roster ID.
# Roster IDs can repeat between seasons, so this key allows both years
# to exist together in the same data model.
dim_roster["roster_key"] = (
    dim_roster["season"].astype(str)
    + "_"
    + dim_roster["roster_id"].astype(str)
)


# Preview roster dimension with newly created key
dim_roster.head()

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 57, Finished, Available, Finished, False)

,season,roster_id,owner_id,roster_key
0,2024,1,600417631085854720,2024_1
1,2024,2,600732404100956160,2024_2
2,2024,3,734418867744526336,2024_3
3,2024,4,999125378960617472,2024_4
4,2024,5,730546204496211968,2024_5


In [34]:
#save dim_roster as a delta table
spark_df = spark.createDataFrame(dim_roster)

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Dim_Roster")

StatementMeta(, e02891e7-91dd-487e-845e-b41f1fb247ec, 58, Finished, Available, Finished, False)